# 🎙️ IndexTTS2 Lightweight English Model Training on Kaggle

Welcome! This notebook is configured to run on Kaggle's free GPUs (P100 or T4 x2). 
Make sure you head to the sidebar (Settings / Accelerator) and turn on a GPU before starting.

### Step 1: Clone Your Repository
We will pull your exact code and lightweight setup directly from GitHub.

In [ ]:
!git clone https://github.com/dipankarhandique340/IndexTTS3.git
%cd IndexTTS3

### Step 2: Set Up Environment & Requirements
We will install the required PyTorch packages, DeepSpeed for optimized training, and other audio dependencies.

In [ ]:
# Install Flash Attention & dependencies
!pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install flash-attn --no-build-isolation
!pip install deepspeed
!pip install uv
!pip uninstall -y tensorflow tensorboard
!sed -i '/flash-attn = { path =/d' pyproject.toml
!rm -f uv.lock
!uv pip install --system -e .

### Step 3: Generate the Fresh Lightweight Checkpoints
Since the large `.pth` checkpoint files are ignored by GitHub, we need to quickly re-generate your empty, lightweight Neural Network architecture right here in Kaggle so we have a blank slate to train on.

In [ ]:
!cd training_lightweight_en && python init_model.py
!ls -lh training_lightweight_en/*.pth

### Step 3.5: Skip Preprocessing & Download Fast Path Backup! ⚡
If you already spent an hour running Step 4 & 4.5 in a previous Kaggle session, you DO NOT need to do it again forever.
Simply run this cell to download your lightning fast processed backup folder, and instantly skip down to **Step 5**!

In [ ]:
import os
# Paste your VikingFile Direct Download Link inside these quotes!
# e.g. "https://vikingfile.com/d/TPRSfLvcIu"
FAST_BACKUP_URL = "https://vikingfile.com/d/7J1TUPYn5Z/en_processed_data.zip"

print("📥 Downloading ultra-fast dataset backup...")
!mkdir -p datasets
!wget -O datasets/en_processed_data.zip $FAST_BACKUP_URL

print("📦 Unzipping 1-Hour processed data back into Kaggle...")
!unzip -q datasets/en_processed_data.zip -d ./
print("✅ DONE! YOU MAY NOW SKIP EXACTLY TO STEP 5: TRAIN GPT!")

### Step 3.6: Fix Dataset Pathing Bug ⚠️
Because of how ZIP files encapsulate their own parent folders, extracting the backup often creates a deeper `datasets/datasets/...` folder than intended. Run this block to instantly fix the paths!

In [ ]:
# Always download tokenizer & stats (required for training)
!mkdir -p checkpoints
!huggingface-cli download IndexTeam/IndexTTS-2 bpe.model wav2vec2bert_stats.pt config.yaml --local-dir checkpoints

# Fix nested datasets path from ZIP extraction
import os
!mkdir -p datasets/en_processed_data
!mv datasets/datasets/en_processed_data/* datasets/en_processed_data/ 2>/dev/null || true
!rm -rf datasets/datasets
!echo 'Tokenizer downloaded & paths fixed!'

### Step 3.7: Training Mode Selection
Set which models to train. If GPT is already done, create the skip flag.

In [ ]:
# ===== TRAINING MODE =====
# Set to True if GPT is already trained (skip Step 5)
SKIP_GPT = True   # <-- Change to False if you need to train GPT
TRAIN_S2MEL = True  # <-- Change to False to skip S2Mel

import os
if SKIP_GPT:
    open('SKIP_GPT_TRAINING', 'w').close()
    print('GPT training will be SKIPPED')
else:
    if os.path.exists('SKIP_GPT_TRAINING'):
        os.remove('SKIP_GPT_TRAINING')
    print('GPT training will RUN')

### Step 4: Download & Extract Your Dataset (LJSpeech)
We will be training the English TTS model using the legendary LJSpeech dataset.
1. The script below automatically downloads the `.tar.bz2` dataset directly from keithito.com.
2. It automatically extracts it to `datasets/LJSpeech-1.1`.

In [ ]:
import os
if os.path.exists('datasets/en_processed_data/train_manifest.jsonl'):
    print('\u2705 Processed data already exists from Step 3.5! Skipping download...')
else:
    DATASET_URL="https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"
    !mkdir -p datasets
    !wget -O datasets/ljspeech.tar.bz2 $DATASET_URL
    !tar -xf datasets/ljspeech.tar.bz2 -C datasets/
    !echo '\u2705 LJSpeech Dataset downloaded and extracted!'
    !ls -lh datasets/LJSpeech-1.1

### Step 4.5: Preprocess LJSpeech
IndexTTS doesn't understand raw audio; it needs mathematical embeddings. 
1. First, we download the official BPE Tokenizer and W-BERT stats.
2. We will convert the `metadata.csv` into a `.jsonl` manifest.
3. We will run `preprocess_data.py` to extract features.
4. We run `generate_gpt_pairs.py` to pair them up.

In [ ]:
import os
if os.path.exists('datasets/en_processed_data/train_manifest.jsonl'):
    print('\u2705 Data already exists! Skipping HF download (already done in Step 3.6)...')
else:
    !mkdir -p checkpoints
    !huggingface-cli download IndexTeam/IndexTTS-2 bpe.model wav2vec2bert_stats.pt config.yaml --local-dir checkpoints

In [ ]:
import os
if os.path.exists('datasets/en_processed_data/train_manifest.jsonl'):
    print('\u2705 Processed data already exists! Skipping CSV conversion...')
else:
    import csv
    import json
    csv_path = 'datasets/LJSpeech-1.1/metadata.csv'
    jsonl_path = 'datasets/LJSpeech-1.1/manifest.jsonl'
    audio_dir = 'datasets/LJSpeech-1.1/wavs'
    print('Converting metadata.csv to IndexTTS manifest.jsonl...')
    with open(csv_path, 'r', encoding='utf-8') as f_in, open(jsonl_path, 'w', encoding='utf-8') as f_out:
        reader = csv.reader(f_in, delimiter='|')
        for row in reader:
            if len(row) < 2: continue
            file_id = row[0]
            text = row[2] if len(row) == 3 and row[2] else row[1]
            entry = {'id': file_id, 'audio': f'{audio_dir}/{file_id}.wav', 'text': text, 'speaker': 'LJ', 'language': 'en'}
            f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print('Conversion complete!')

In [ ]:
import os
if os.path.exists('datasets/en_processed_data/train_manifest.jsonl'):
    print('\u2705 Processed data already exists! Skipping 1-hour preprocessing...')
else:
    !python tools/preprocess_data.py \\
        --manifest datasets/LJSpeech-1.1/manifest.jsonl \\
        --output-root datasets \\
        --tokenizer checkpoints/bpe.model \\
        --config training_lightweight_en/config_light.yaml \\
        --gpt-checkpoint training_lightweight_en/gpt_light_init.pth \\
        --language en \\
        --workers 2

In [ ]:
import os
if os.path.exists('datasets/en_processed_data/gpt_pairs_train.jsonl'):
    print('\u2705 GPT pairs already exist! Skipping generation...')
else:
    !python tools/generate_gpt_pairs.py \\
        --dataset datasets/en_processed_data

### Step 4.6: Backup Filtered Dataset to Cloud ☁️
This zips the exact dataset we just spent 1 hour processing and pushes it to your VikingFile server. If Kaggle shuts down tomorrow, you can download the Zip instead of processing!

In [ ]:
import os
import requests

print("📦 Zipping the dataset folder (This may take a few minutes)...")
os.system("zip -q -r datasets/en_processed_data.zip datasets/en_processed_data")
print("✅ Zip complete. Requesting VikingFile server...")

server = requests.get("https://vikingfile.com/api/get-server").json()["server"]
print(f"🚀 Uploading Backup to {server}...")

# Using cURL to prevent memory crashes with huge file uploads
cmd = f'curl -# -F "file=@datasets/en_processed_data.zip" -F "user=yUgBAcnoF7" {server}'
os.system(cmd)
print("
🏆 Backup successfully saved to your Cloud account!")

### Step 5: Train the Lightweight GPT (Text-to-Semantic)
This begins the training for the ~1.7 GB GPT component.

In [ ]:
import os
if not os.path.exists('SKIP_GPT_TRAINING'):
    pass  # writefile cell runs regardless, skip is in the bash cell
%%writefile training_lightweight_en/train_lightweight_gpt.sh
#!/bin/bash
echo "Starting GPT lightweight TURBO training..."
python trainers/train_gpt_v2.py \
    --train-manifest datasets/en_processed_data/gpt_pairs_train.jsonl \
    --val-manifest datasets/en_processed_data/gpt_pairs_val.jsonl \
    --tokenizer checkpoints/bpe.model \
    --config training_lightweight_en/config_light.yaml \
    --base-checkpoint training_lightweight_en/gpt_light_init.pth \
    --output-dir checkpoints_lightweight_gpt \
    --resume auto \
    --batch-size 64 \
    --grad-accumulation 2 \
    --epochs 15 \
    --learning-rate 3e-4 \
    --weight-decay 0.01 \
    --warmup-steps 200 \
    --log-interval 5 \
    --val-interval 1000 \
    --grad-clip 1.0 \
    --text-loss-weight 0.2 \
    --mel-loss-weight 0.8 \
    --amp

In [ ]:
import os
if os.path.exists('checkpoints_lightweight_gpt/latest.pth'):
    print('GPT model already trained! Skipping Step 5...')
else:
    !chmod +x training_lightweight_en/train_lightweight_gpt.sh
    !bash training_lightweight_en/train_lightweight_gpt.sh

### Step 6: Train the Lightweight DiT / S2Mel Component
Once the GPT model finishes training, you'll need to train S2Mel to accurately convert those semantic tokens into raw Mel Spectrograms.

In [ ]:
# S2Mel needs raw audio for mel extraction
# Download LJSpeech if not already present
import os
if not os.path.exists('datasets/LJSpeech-1.1/wavs'):
    print('Downloading LJSpeech audio for S2Mel training...')
    !wget -q -O datasets/ljspeech.tar.bz2 https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xf datasets/ljspeech.tar.bz2 -C datasets/
    !rm datasets/ljspeech.tar.bz2
    print('LJSpeech downloaded!')
else:
    print('LJSpeech audio already exists!')

In [ ]:
%%writefile training_lightweight_en/train_lightweight_s2mel.sh
#!/bin/bash
echo "Starting S2Mel lightweight TURBO training..."
python trainers/train_s2mel_v2.py \
    --train-manifest datasets/en_processed_data/train_manifest.jsonl \
    --val-manifest datasets/en_processed_data/val_manifest.jsonl \
    --config training_lightweight_en/config_light.yaml \
    --base-checkpoint training_lightweight_en/s2mel_light_init.pth \
    --output-dir checkpoints_lightweight_s2mel \
    --audio-root datasets/LJSpeech-1.1/wavs \
    --resume auto \
    --batch-size 32 \
    --grad-accumulation 2 \
    --epochs 30 \
    --learning-rate 5e-4 \
    --val-interval 1000 \
    --amp

In [ ]:
!chmod +x training_lightweight_en/train_lightweight_s2mel.sh
!bash training_lightweight_en/train_lightweight_s2mel.sh

### Step 7: Auto-Backup Trained Models to Cloud ☁️
This automatically zips both trained model checkpoints and uploads them to your VikingFile cloud account.
Even if Kaggle deletes the session files, your models are safely stored forever!

In [ ]:
import os
import requests

print("\ud83d\udce6 Zipping both trained model folders...")
os.system("zip -q -r trained_models_backup.zip checkpoints_lightweight_gpt checkpoints_lightweight_s2mel")
print("\u2705 Zip complete!")

# Get file size
size_mb = os.path.getsize('trained_models_backup.zip') / (1024*1024)
print(f"\ud83d\udcc1 Backup size: {size_mb:.1f} MB")

# Upload to VikingFile
print("\ud83d\ude80 Uploading to VikingFile...")
server = requests.get('https://vikingfile.com/api/get-server').json()['server']
cmd = f'curl -# -F "file=@trained_models_backup.zip" -F "user=yUgBAcnoF7" {server}'
os.system(cmd)
print("\n\ud83c\udfc6 Both GPT + S2Mel models safely backed up to your VikingFile account!")
print("\ud83d\udd17 Log into https://vikingfile.com to find your trained_models_backup.zip")

### Congratulations! 🎉
You have successfully trained your lightweight voice clones. Save the output models located in `checkpoints_lightweight_gpt` and `checkpoints_lightweight_s2mel` (found inside the `/kaggle/working/IndexTTS3/` output panel on the right) back to your local computer.